<a href="https://colab.research.google.com/github/nachopedretti/diploup2026-TP2/blob/main/notebooks/03_preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Preprocesamiento

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

Carga de datasets

In [6]:
df_etiq = pd.read_csv("https://raw.githubusercontent.com/nachopedretti/diploup2026-TP2/main/data/raw/smoking_prediction.csv")
df_pred = pd.read_csv("https://raw.githubusercontent.com/nachopedretti/diploup2026-TP2/main/data/raw/smoking_prediction_entrega.csv")
print("Datos etiquetados:", df_etiq.shape)
print("Datos para predecir:", df_pred.shape)

Datos etiquetados: (50000, 27)
Datos para predecir: (5692, 26)


Ver variables categoricas

In [7]:
for columna in ["gender", "tartar", "oral"]:
    print(columna, df_etiq[columna].unique())
print("\nValores faltantes:")
print("Etiquetados:", df_etiq.isna().sum().sum())
print("Predicción:", df_pred.isna().sum().sum())

gender ['F' 'M']
tartar ['Y' 'N']
oral ['Y']

Valores faltantes:
Etiquetados: 0
Predicción: 0


Separamos la variable oral smoking de las variables para usar para predecir tambien oral dado q tiene un unico valor.

In [8]:
X = df_etiq.drop(columns=["smoking", "ID", "oral"]).copy()
y = df_etiq["smoking"].astype(int).copy()
X_pred = df_pred.drop(columns=["ID", "oral"]).copy()
print("Variables predictoras:", X.shape)
print("Variable objetivo:", y.shape)
print("Datos para predecir:", X_pred.shape)
print("\nMismas columnas:", X.columns.equals(X_pred.columns))

Variables predictoras: (50000, 24)
Variable objetivo: (50000,)
Datos para predecir: (5692, 24)

Mismas columnas: True


**División de Train y Test**

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,
    random_state=42,stratify=y)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nDistribución original:")
print((y.value_counts(normalize=True) * 100).round(2))

print("\nDistribución Train:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nDistribución Test:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train: (40000, 24)
Test: (10000, 24)

Distribución original:
smoking
0    63.34
1    36.66
Name: proportion, dtype: float64

Distribución Train:
smoking
0    63.34
1    36.66
Name: proportion, dtype: float64

Distribución Test:
smoking
0    63.34
1    36.66
Name: proportion, dtype: float64


In [10]:
def distribucion_genero_smoking(X_datos, y_datos, nombre):
    tabla = pd.DataFrame({
        "gender": X_datos["gender"].to_numpy(),
        "smoking": y_datos.to_numpy()
    })

    tabla["gender"] = tabla["gender"].map({
        "F": "Mujeres",
        "M": "Hombres"
    })

    resultado = pd.crosstab(
        tabla["gender"],
        tabla["smoking"],
        normalize="index"
    ) * 100

    resultado.columns = ["No fumadores (%)", "Fumadores (%)"]

    print(f"\n{nombre}")
    display(resultado.round(2))


distribucion_genero_smoking(X, y, "Original")
distribucion_genero_smoking(X_train, y_train, "Train")
distribucion_genero_smoking(X_test, y_test, "Test")


Original


,No fumadores (%),Fumadores (%)
gender,,
Hombres,44.78,55.22
Mujeres,95.72,4.28



Train


,No fumadores (%),Fumadores (%)
gender,,
Hombres,44.79,55.21
Mujeres,95.65,4.35



Test


,No fumadores (%),Fumadores (%)
gender,,
Hombres,44.75,55.25
Mujeres,96.03,3.97


Dividimos los datos en 80% Train y 20% Test, utilizando una división stratify para conservar la distribución original de la variable **smoking** (63,34% no fumadores y 36,66% fumadores).

**Transformación de variables categóricas:**
 Convertimos gender y tartar a valores binarios para que puedan utilizarse en los modelos de clasificación.

In [11]:
def transformar_categoricas(datos):
    datos = datos.copy()
    datos["gender"] = datos["gender"].map({"F": 0, "M": 1})
    datos["tartar"] = datos["tartar"].map({"N": 0, "Y": 1})
    return datos
X_train_pre = transformar_categoricas(X_train)
X_test_pre = transformar_categoricas(X_test)
X_pred_pre = transformar_categoricas(X_pred)

In [12]:
for nombre, datos in [
    ("Train", X_train_pre),
    ("Test", X_test_pre),
    ("Predicción", X_pred_pre)
]:
    print(nombre)
    print("Valores faltantes:", datos.isna().sum().sum())
    print("Columnas de texto:", datos.select_dtypes(include="object").columns.tolist())
    print()

print("Ejemplo de transformación:")
print(X_train_pre[["gender", "tartar"]].head())

Train
Valores faltantes: 0
Columnas de texto: []

Test
Valores faltantes: 0
Columnas de texto: []

Predicción
Valores faltantes: 0
Columnas de texto: []

Ejemplo de transformación:
       gender  tartar
7883        0       1
3451        0       0
3495        1       0
49723       1       0
34452       1       0


**Escalado de variables numéricas** usamos MinMaxScaler para normalizar las variables numéricas continuas

In [13]:
columnas_sin_escalar = [
    "gender",
    "tartar",
    "dental caries",
    "hearing(left)",
    "hearing(right)",
    "Urine protein"]

columnas_escalar = [
    col for col in X_train_pre.columns
    if col not in columnas_sin_escalar]

print("Variables a escalar:", len(columnas_escalar))
print(columnas_escalar)

Variables a escalar: 18
['age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'serum creatinine', 'AST', 'ALT', 'Gtp']


In [14]:
scaler = MinMaxScaler()

X_train_final = X_train_pre.copy()
X_test_final = X_test_pre.copy()
X_pred_final = X_pred_pre.copy()

scaler.fit(X_train_pre[columnas_escalar])

X_train_final[columnas_escalar] = scaler.transform(
    X_train_pre[columnas_escalar])

X_test_final[columnas_escalar] = scaler.transform(
    X_test_pre[columnas_escalar])

X_pred_final[columnas_escalar] = scaler.transform(
    X_pred_pre[columnas_escalar])

Utilizamos fit solamente con Train



In [15]:
for nombre, datos in [
    ("Train", X_train_final),
    ("Test", X_test_final),
    ("Predicción", X_pred_final)]:
    print(nombre, datos.shape)
    print("Valores faltantes:", datos.isna().sum().sum())
    print("Columnas de texto:", datos.select_dtypes(include="object").columns.tolist())
print("\nRangos de las variables escaladas en Train:")
print(X_train_final[columnas_escalar].agg(["min", "max"]).round(2))

Train (40000, 24)
Valores faltantes: 0
Columnas de texto: []
Test (10000, 24)
Valores faltantes: 0
Columnas de texto: []
Predicción (5692, 24)
Valores faltantes: 0
Columnas de texto: []

Rangos de las variables escaladas en Train:
     age  height(cm)  weight(kg)  waist(cm)  eyesight(left)  eyesight(right)  \
min  0.0         0.0         0.0        0.0             0.0              0.0   
max  1.0         1.0         1.0        1.0             1.0              1.0   

     systolic  relaxation  fasting blood sugar  Cholesterol  triglyceride  \
min       0.0         0.0                  0.0          0.0           0.0   
max       1.0         1.0                  1.0          1.0           1.0   

     HDL  LDL  hemoglobin  serum creatinine  AST  ALT  Gtp  
min  0.0  0.0         0.0               0.0  0.0  0.0  0.0  
max  1.0  1.0         1.0               1.0  1.0  1.0  1.0  


**Guardado de datos**

In [18]:
from pathlib import Path
import joblib

# Carpetas de salida en Colab
ruta_datos = Path("/content/data/processed")
ruta_modelos = Path("/content/models")

ruta_datos.mkdir(parents=True, exist_ok=True)
ruta_modelos.mkdir(parents=True, exist_ok=True)

# Guardar Train, Test y Predicción
X_train_final.to_csv(ruta_datos / "X_train.csv", index=False)
X_test_final.to_csv(ruta_datos / "X_test.csv", index=False)
X_pred_final.to_csv(ruta_datos / "X_pred.csv", index=False)

# Guardar variable objetivo
y_train.to_csv(ruta_datos / "y_train.csv", index=False)
y_test.to_csv(ruta_datos / "y_test.csv", index=False)

# Guardar escalador
joblib.dump(
    {"scaler": scaler, "columnas_escalar": columnas_escalar},
    ruta_modelos / "preprocesamiento.joblib")

print("Datos guardados en:", ruta_datos.resolve())
print("Escalador guardado en:", ruta_modelos.resolve())

Datos guardados en: /content/data/processed
Escalador guardado en: /content/models


In [17]:
for archivo in sorted(ruta_datos.glob("*.csv")):
    print(archivo.name)

print("\nEscalador guardado:",
      (ruta_modelos / "preprocesamiento.joblib").exists())

X_pred.csv
X_test.csv
X_train.csv
y_test.csv
y_train.csv

Escalador guardado: True


## Conclusiones

- Preparamos los datos para el entrenamiento de modelos de clasificación, eliminando las variables **ID** y **oral** por no aportar información predictiva, la primer a es el id de persona y el segundo tiene el mismo valor en todo el dataset.

- Dividimos los registros etiquetados en 80% Train y 20% Test, manteniendo la proporción original de fumadores y no fumadores.

- Transformamos las variables categóricas y aplicamos **MinMaxScaler** a 18 variables numéricas continuas.

- Verificamos que los conjuntos procesados contienen 24 variables numéricas y no presentan valores faltantes. Utilizamos fit solamente con Train

- Los datos quedaron preparados para continuar con el entrenamiento y la comparación de modelos.